In [4]:
#Rank-based linear factor model for SP500
import pandas as pd
import yfinance as yf
import requests

#Getting top 500 companies from wiki
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
}

html = requests.get(url, headers=headers).text

sp500 = pd.read_html(html)[0]
tickers = sp500["Symbol"].str.replace(".", "-", regex=False).tolist()  # BRK.B -> BRK-B for yfinance

data = [] 

#Storing the companies' data in a list
for t in tickers:
    try:
        info = yf.Ticker(t).info
        data.append({
            "ticker": t,
            "earnings_yield": info.get("trailingEps", None) / info.get("priceToBook", None),
            "roc": info.get("returnOnAssets", None),
            "volatility": info.get("beta", None)
        })
    except:
        pass

#Ranking the data
df = pd.DataFrame(data).dropna()
df["rank_earnings_yield"] = df["earnings_yield"].rank(ascending=False)
df["rank_roc"] = df["roc"].rank(ascending=False)
df["rank_volatility"] = df["volatility"].rank(ascending=True)

#Creating the ranking values for the algo
signs = {
    "rank_earnings_yield": +1,
    "rank_roc": +1,
    "rank_volatility": -1
}

#Calculating the final Ranks for the companies
df["final_rank"] = sum(signs[col] * df[col] for col in signs)

df = df.sort_values("final_rank", ascending=False)

top_30 = df.head(30)
print(top_30[["ticker", "final_rank"]])


C:\Users\suley\AppData\Local\Temp\ipykernel_41016\1170397464.py:15: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  sp500 = pd.read_html(html)[0]


    ticker  final_rank
93     CNC       868.0
274    KHC       836.0
154    DOW       791.0
413    SJM       786.0
59     BAX       768.0
10     APD       764.0
292    LYB       763.5
432   TTWO       749.0
317    TAP       746.5
127   CSGP       741.0
472   VTRS       732.5
134    CVS       718.0
457    UDR       717.0
303    MCK       714.0
85     CAH       710.5
445    TKO       694.0
314   MRNA       676.0
466    VTR       670.0
92     COR       670.0
251     IP       665.0
250    IFF       651.0
130   CRWD       644.5
365    PCG       635.5
23    AMCR       631.0
79     BXP       628.5
356   PANW       627.0
232    HRL       612.0
94     CNP       605.5
487   WELL       604.0
224    DOC       598.0


Hypothesis Testing Steps

1. Based on a backtest on some finite sample of data, we compute a certain statistical measure called the test statistic. For concreteness, lets say the test statistic is the average daily return of a trading strategy in that period.

2. We suppose that the true average daily return based on an infinite data set is actually zero. This supposition is called the null hypothesis.

3. We suppose that the probability distribution of daily returns is known. The probability distribution has a zero mean, based on the null hypothesis. We describe later how we determine this probability distribution.

4. Based on this null hypothesis probability distribution, we compute the probability p that the average daily returns will be at least as large as the observered value in the backtest. This probability p is called the p-value and if it is very small (smaller than 0.01), that means we can "reject the null hypothesis", and conclude that the backtested average daily return is statistically significant.

## Regime Identification: Mean Reversion vs Trend

Before applying any trading strategy, it is essential to understand the statistical behavior of a price series.

---

### 1. Augmented Dickey–Fuller (ADF) Test
The ADF test checks whether a price series contains a unit root, which would indicate random-walk behavior. This test is based on the regression: Δy_t = α + λ y_{t-1} + ε_t

---

The null hypothesis is \( \lambda = 0 \), meaning that price changes are independent of the current price level. Rejecting this hypothesis suggests the series is **stationary**, which is a necessary condition for mean reversion.

In practice, individual stocks often fail the ADF test due to structural changes, growth, and regime shifts. Therefore, the ADF test is used here primarily as a **statistical reference**, not as a strict trading filter.

### 2. Lambda (λ) and Half-Life of Mean Reversion
Even when the ADF test does not reject the random-walk hypothesis, the estimated value of \( \lambda \) still provides valuable economic insight. The coefficient \( \lambda \) measures how strongly price changes respond to deviations from the mean:

- \( \lambda < 0 \): mean-reverting behavior  
- \( \lambda > 0 \): trending behavior  

Interpreting the regression as an Ornstein–Uhlenbeck process allows us to compute the **half-life of mean reversion**:

\[
\text{Half-life} = \frac{-\ln(2)}{\lambda}
\]

The half-life represents the expected time for a price deviation to decay by 50%. Short half-lives indicate strong and potentially tradable mean reversion, while long half-lives suggest that mean-reversion strategies may be impractical.


### 3. Hurst Exponent
The Hurst exponent measures how the variance of price changes scales with time:

\[
\mathrm{Var}[z(t+\tau) - z(t)] \sim \tau^{2H}
\]

Interpretation:
- \( H = 0.5 \): random walk  
- \( H < 0.5 \): mean reversion  
- \( H > 0.5 \): trending behavior  

Unlike the ADF test, the Hurst exponent captures **long-range dependence** and provides a complementary, scale-based view of market behavior.

### 4. Strategy Interpretation
No single test is decisive on its own. Instead, these diagnostics should be interpreted together:

- **ADF**: statistical evidence against random walk (strict but informative)
- **λ and half-life**: economic usefulness and trading time scale
- **Hurst exponent**: persistence vs anti-persistence across horizons

If λ is negative with a reasonably short half-life and the Hurst exponent is below 0.5, a **mean-reversion strategy** may be appropriate. If λ is positive or the Hurst exponent is above 0.5, a **trend-following strategy** is more suitable. When signals are weak or contradictory, the asset may be close to a random walk, and caution is warranted.

---

### Key Takeaway
The goal of these tests is not to “prove” a strategy, but to **avoid applying the wrong strategy to the wrong market regime**. Understanding regime behavior is a critical first step in systematic trading and helps guide both strategy selection and parameter choices.

In [14]:
#Running ADF, Hurst, and Mean-life test to determine what strategy
#to decide between Mean Reversion vs Trend strategies 

import numpy as np
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller

ticker = "AAPL"
prices = yf.download(ticker,start="2018-01-01", auto_adjust=True, progress=False)["Close"].dropna()
logp = np.log(prices)

print(f"{ticker} data points:", len(logp))

#ADF TEST
# PURPOSE:
# The ADF test checks whether the price series behaves like
# a RANDOM WALK (unit root) or is STATIONARY.
#
# Model tested (simplified):
# Δy_t = λ y_{t-1} + μ + ε_t
#
# Null hypothesis (H0):
#   λ = 0  → random walk (no mean reversion)
#
# Alternative hypothesis (H1):
#   λ < 0 → mean reverting
#
# We test this using the t-statistic of λ against
# Dickey-Fuller critical values.

adf_stat, adf_p, adf_lags, adf_nobs, adf_crit = adfuller(
    logp, maxlag=1, regression="c", autolag=None
)

print("\n--- ADF Test on log(price) ---")
print("ADF statistic:", adf_stat)
print("p-value:", adf_p)
print("lags used:", adf_lags)
print("critical values:", adf_crit)

# INTERPRETATION:
# If ADF statistic < critical value → reject random walk
# In practice, single stocks usually FAIL ADF

#Lambda + Half-life calculation
# PURPOSE:
# Even if ADF fails, traders still want to know:
# "How fast does the price mean-revert?"
#
# We estimate λ directly using the regression:
#
# Δy_t = α + λ y_{t-1} + ε_t
#
# λ < 0 → mean reversion
# λ > 0 → trending behavior
#
# From Ornstein-Uhlenbeck theory, the HALF-LIFE of mean
# reversion is:
#
#   half_life = -ln(2) / λ

# First difference: Δy_t = y_t - y_{t-1}
dy = logp.diff().dropna()
y_lag = logp.shift(1).dropna()
dy = dy.loc[y_lag.index]

X = sm.add_constant(y_lag)
ols = sm.OLS(dy,X).fit()

lambda_hat = ols.params[1]
lambda_se = ols.bse[1]
lambda_t = lambda_hat / lambda_se

half_life = (-np.log(2) / lambda_hat) if lambda_hat < 0 else np.inf

print("\n--- Lambda (λ) + Half-life ---")
print("lambda_hat:", lambda_hat)
print("SE(lambda):", lambda_se)
print("t-stat (lambda/SE):", lambda_t)
print("half_life (days):", half_life)

# INTERPRETATION:
# λ > 0        → trending (momentum strategies)
# λ < 0        → mean reverting
# half-life:
#   < 20 days  → strong mean reversion
#   20–50 days → weak but tradable
#   > 100 days → impractical

#Hurst exponent
# PURPOSE:
# Measures how variance scales with time:
#
# Var[z(t+τ) - z(t)] ~ τ^(2H)
#
# H = 0.5 → random walk
# H < 0.5 → mean reversion
# H > 0.5 → trending
#
# This is a COMPLEMENT to ADF (not a replacement).
def hurst_exponent(x, max_lag=100):
    x = np.asarray(x).reshape(-1)  # force 1D, avoids (N,1) shape issues too
    max_lag = min(max_lag, len(x) - 2)

    lags = range(2, max_lag + 1)

    # RMS of lagged differences: sqrt(E[(x_t - x_{t-lag})^2])
    tau = [np.sqrt(np.mean((x[lag:] - x[:-lag])**2)) for lag in lags]

    # slope of log(tau) vs log(lag) is H
    poly = np.polyfit(np.log(list(lags)), np.log(tau), 1)
    return poly[0]


H = hurst_exponent(logp.values, max_lag=100)
print("\n--- Hurst Exponent ---")
print("H:", H)

# INTERPRETATION:
# H < 0.5 → mean reverting
# H ≈ 0.5 → random walk
# H > 0.5 → trending


print("\n--- Quick Interpretation ---")
if lambda_hat > 0:
    print("λ > 0 → trending bias (momentum/trend-following more appropriate).")
elif np.isfinite(half_life) and half_life < 30:
    print("λ < 0 with half-life < ~30 days → mean reversion may be tradable.")
else:
    print("λ ≤ 0 but half-life is long or λ ~ 0 → weak mean reversion / near random walk.")
print("H < 0.5 suggests mean reversion; H > 0.5 suggests trending; H ≈ 0.5 suggests random walk.")
print("ADF p-value small (e.g., <0.05) would support stationarity, but stocks often fail ADF.")

AAPL data points: 2001

--- ADF Test on log(price) ---
ADF statistic: -1.0365217361225296
p-value: 0.7397119407738956
lags used: 1
critical values: {'1%': np.float64(-3.4336254962865045), '5%': np.float64(-2.862986937508278), '10%': np.float64(-2.567540287745173)}

--- Lambda (λ) + Half-life ---
lambda_hat: -0.0007642383195376938
SE(lambda): 0.000722459821647146
t-stat (lambda/SE): -1.0578281264074956
half_life (days): 906.9777880010606

--- Hurst Exponent ---
H: 0.5263873156187701

--- Quick Interpretation ---
λ ≤ 0 but half-life is long or λ ~ 0 → weak mean reversion / near random walk.
H < 0.5 suggests mean reversion; H > 0.5 suggests trending; H ≈ 0.5 suggests random walk.
ADF p-value small (e.g., <0.05) would support stationarity, but stocks often fail ADF.


C:\Users\suley\AppData\Local\Temp\ipykernel_41016\1280639043.py:70: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  lambda_hat = ols.params[1]
C:\Users\suley\AppData\Local\Temp\ipykernel_41016\1280639043.py:71: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  lambda_se = ols.bse[1]
